# MODELOS IA

## Usuarios vs Servidores - Mes completo (MLP)

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.utils import resample
import scipy.stats as st
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================

# [CONFIGURACIÓN DE USUARIO]: Define las rutas
BASE_DIR = "ruta/a/tu/espacio/de/trabajo"
INPUT_DIR = os.path.join(BASE_DIR, "caracteristicas")
OUTPUT_DIR = os.path.join(BASE_DIR, "resultados_mensual_mlp")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# [CONFIGURACIÓN DE USUARIO]: Define explícitamente las etiquetas que corresponden a los servidores.
ETIQUETAS_SERVIDORES = [
    'etiqueta_servidores_1', 
    'etiqueta_servidores_2'
]

N_FOLDS = 10  # Número de ejecuciones para rigor estadístico

# ==========================================
# 2. FUNCIÓN DE CARGA Y BALANCEO
# ==========================================
def cargar_y_balancear(metrica, wavelet):
    """Carga el dataset único, etiqueta y balancea (Undersampling)."""
    archivo = f"dataset_{metrica}_{wavelet}.csv"
    ruta = os.path.join(INPUT_DIR, archivo)
    
    if not os.path.exists(ruta):
        return None, None

    df = pd.read_csv(ruta)
    
    # Crear target binario (1 = Servidor, 0 = Usuario)
    df['target'] = df['etiqueta'].apply(lambda x: 1 if x in ETIQUETAS_SERVIDORES else 0)
    
    # Balanceo (Undersampling)
    df_u = df[df['target'] == 0]
    df_s = df[df['target'] == 1]
    
    n_min = min(len(df_u), len(df_s))
    if n_min < 10: 
        return None, None
        
    df_u_bal = resample(df_u, replace=False, n_samples=n_min, random_state=42)
    df_s_bal = resample(df_s, replace=False, n_samples=n_min, random_state=42)
    
    df_final = pd.concat([df_u_bal, df_s_bal], ignore_index=True)
    # Mezclamos aleatoriamente
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    
    features = [c for c in df_final.columns if c.startswith('nivel_') or c.startswith('escala_')]
    features.sort()
    
    return df_final[features].values, df_final['target'].values

# ==========================================
# 3. EJECUCIÓN DEL BENCHMARK ESTADÍSTICO
# ==========================================
def ejecutar_benchmark_riguroso():
    print("=== INICIANDO BENCHMARK MENSUAL (10-FOLD CV) - MLP ===")
    
    metricas = ["energia", "entropia"]
    wavelets = ["haar", "db2", "mexicanhat"]
    
    modelos = {
        '1_Baseline_Aleatorio': DummyClassifier(strategy='stratified', random_state=42),
        '2_Baseline_Lineal': LogisticRegression(random_state=42, max_iter=1000),
        '3_MLP_Propuesto': MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam', max_iter=1000, random_state=42)
    }

    resultados_globales = []

    for metrica in metricas:
        for wav in wavelets:
            X, y = cargar_y_balancear(metrica, wav)
            if X is None:
                continue
                
            print(f"\n[{metrica.upper()} + {wav.upper()}] Evaluando...")
            print(f"  -> Instancias balanceadas: {len(y)} | Usuarios: {sum(y==0)} | Servidores: {sum(y==1)}")
            
            datos_roc_config = {}
            
            for nombre_modelo, modelo in modelos.items():
                skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
                
                f1_scores, acc_scores = [], []
                prec_c0, rec_c0, f1_c0 = [], [], []
                prec_c1, rec_c1, f1_c1 = [], [], []
                
                y_true_oof, y_prob_oof = [], []
                cm_acumulada = np.zeros((2, 2), dtype=int)
                
                for train_idx, test_idx in skf.split(X, y):
                    X_train, X_test = X[train_idx], X[test_idx]
                    y_train, y_test = y[train_idx], y[test_idx]
                    
                    # Escalar dinámicamente para evitar Data Leakage
                    scaler = StandardScaler()
                    X_train_sc = scaler.fit_transform(X_train)
                    X_test_sc = scaler.transform(X_test)
                    
                    import sklearn
                    modelo_clon = sklearn.base.clone(modelo)
                    modelo_clon.fit(X_train_sc, y_train)
                    y_pred = modelo_clon.predict(X_test_sc)
                    
                    y_prob = modelo_clon.predict_proba(X_test_sc)[:, 1]
                    y_true_oof.extend(y_test)
                    y_prob_oof.extend(y_prob)
                    
                    acc_scores.append(accuracy_score(y_test, y_pred))
                    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1], zero_division=0)
                    
                    prec_c0.append(p[0]); rec_c0.append(r[0]); f1_c0.append(f[0])
                    prec_c1.append(p[1]); rec_c1.append(r[1]); f1_c1.append(f[1])
                    
                    f1_scores.append(np.mean([f[0], f[1]]))
                    cm_acumulada += confusion_matrix(y_test, y_pred, labels=[0, 1])

                f1_media = np.mean(f1_scores)
                ci = st.t.interval(0.95, df=len(f1_scores)-1, loc=f1_media, scale=st.sem(f1_scores))
                margen_error = ci[1] - f1_media if not np.isnan(ci[1]) else 0

                resultados_globales.append({
                    "Configuracion": f"{metrica.upper()} + {wav.upper()}",
                    "Modelo": nombre_modelo.split('_', 1)[1],
                    "Accuracy": f"{np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}",
                    "F1_Global (95% CI)": f"{f1_media:.4f} ± {margen_error:.4f}",
                    "F1_Usr": f"{np.mean(f1_c0):.4f} ± {np.std(f1_c0):.4f}",
                    "F1_Srv": f"{np.mean(f1_c1):.4f} ± {np.std(f1_c1):.4f}"
                })
                
                fpr, tpr, _ = roc_curve(y_true_oof, y_prob_oof)
                roc_auc = auc(fpr, tpr)
                datos_roc_config[nombre_modelo.split('_', 1)[1]] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}
                
                if nombre_modelo == '3_MLP_Propuesto':
                    plt.figure(figsize=(6, 5))
                    sns.heatmap(cm_acumulada, annot=True, fmt='d', cmap='Purples', cbar=False,
                                xticklabels=['Usuar (0)', 'Serv (1)'], yticklabels=['Usuar (0)', 'Serv (1)'])
                    plt.title(f"Matriz de Confusión Global (10 Folds)\nMLP: {metrica.upper()} + {wav.upper()}")
                    plt.xlabel("Predicción")
                    plt.ylabel("Realidad")
                    plt.tight_layout()
                    plt.savefig(os.path.join(OUTPUT_DIR, f"cm_10fold_MLP_{metrica}_{wav}.pdf"), format='pdf', bbox_inches='tight')
                    plt.close()
                    
                    plt.figure(figsize=(8, 6))
                    plt.plot(modelo_clon.loss_curve_, lw=2, color='darkviolet')
                    plt.title(f"Curva de Convergencia (Loss Curve)\nMLP: {metrica.upper()} + {wav.upper()}")
                    plt.xlabel("Iteraciones (Épocas)")
                    plt.ylabel("Pérdida (Log-Loss)")
                    plt.grid(True, linestyle='--', alpha=0.6)
                    plt.tight_layout()
                    plt.savefig(os.path.join(OUTPUT_DIR, f"loss_curve_mlp_{metrica}_{wav}.pdf"), format='pdf', bbox_inches='tight')
                    plt.close()

            plt.figure(figsize=(8, 6))
            colores_roc = {'Baseline_Aleatorio': 'gray', 'Baseline_Lineal': 'teal', 'MLP_Propuesto': 'darkviolet'}
            for mod_nombre, datos in datos_roc_config.items():
                plt.plot(datos['fpr'], datos['tpr'], lw=2, color=colores_roc.get(mod_nombre, 'blue'),
                         label=f"{mod_nombre} (AUC = {datos['auc']:.4f})")
            
            plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') 
            plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
            plt.xlabel('Tasa de Falsos Positivos (FPR)')
            plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
            plt.title(f'Comparativa Curva ROC: {metrica.upper()} + {wav.upper()}')
            plt.legend(loc="lower right")
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, f"roc_curve_{metrica}_{wav}.pdf"), format='pdf', bbox_inches='tight')
            plt.close()

    # ==========================================
    # 4. EXPORTAR RESULTADOS 
    # ==========================================
    df_resultados = pd.DataFrame(resultados_globales)
    print("\n" + "="*120)
    print("REPORTE ESTADÍSTICO FINAL (MLP)")
    print("="*120)
    print(df_resultados.to_string(index=False))
    
    df_resultados.to_csv(os.path.join(OUTPUT_DIR, "reporte_mensual_mlp.csv"), index=False)
    print(f"\n[INFO] Gráficas y tablas guardadas en: {OUTPUT_DIR}")

if __name__ == "__main__":
    ejecutar_benchmark_riguroso()

## Usuarios vs Servidores - Mes completo (Random Forest)

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.utils import resample
import scipy.stats as st
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================

# [CONFIGURACIÓN DE USUARIO]: Define las rutas
BASE_DIR = "ruta/a/tu/espacio/de/trabajo"
INPUT_DIR = os.path.join(BASE_DIR, "caracteristicas") # ÚNICA CARPETA DE ENTRADA
OUTPUT_DIR = os.path.join(BASE_DIR, "resultados_mensual_rf")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# [CONFIGURACIÓN DE USUARIO]: Define explícitamente las etiquetas que corresponden a los servidores.
ETIQUETAS_SERVIDORES = [
    'etiqueta_servidores_1', 
    'etiqueta_servidores_2'
]

N_FOLDS = 10 

# ==========================================
# 2. FUNCIÓN DE CARGA Y BALANCEO
# ==========================================
def cargar_y_balancear_rf(metrica, wavelet):
    archivo = f"dataset_{metrica}_{wavelet}.csv"
    ruta = os.path.join(INPUT_DIR, archivo)
    
    if not os.path.exists(ruta):
        return None, None, None

    df = pd.read_csv(ruta)
    
    # Crear target binario
    df['target'] = df['etiqueta'].apply(lambda x: 1 if x in ETIQUETAS_SERVIDORES else 0)
    
    # Balanceo (Undersampling)
    df_u = df[df['target'] == 0]
    df_s = df[df['target'] == 1]
    
    n_min = min(len(df_u), len(df_s))
    if n_min < 10: 
        return None, None, None
        
    df_u_bal = resample(df_u, replace=False, n_samples=n_min, random_state=42)
    df_s_bal = resample(df_s, replace=False, n_samples=n_min, random_state=42)
    
    df_final = pd.concat([df_u_bal, df_s_bal], ignore_index=True)
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    
    features = [c for c in df_final.columns if c.startswith('nivel_') or c.startswith('escala_')]
    features.sort()
    
    return df_final[features].values, df_final['target'].values, features

# ==========================================
# 3. EJECUCIÓN DEL BENCHMARK ESTADÍSTICO
# ==========================================
def ejecutar_benchmark_rf():
    print("=== INICIANDO BENCHMARK MENSUAL (10-FOLD CV) - RANDOM FOREST ===")
    
    metricas = ["energia", "entropia"]
    wavelets = ["haar", "db2", "mexicanhat"]
    
    modelos = {
        '1_Baseline_Aleatorio': DummyClassifier(strategy='stratified', random_state=42),
        '2_Baseline_Lineal': LogisticRegression(random_state=42, max_iter=1000),
        '3_RF_Propuesto': RandomForestClassifier(n_estimators=100, criterion='gini', random_state=42, n_jobs=-1)
    }

    resultados_globales = []

    for metrica in metricas:
        for wav in wavelets:
            X, y, nombres_features = cargar_y_balancear_rf(metrica, wav)
            if X is None: continue
                
            print(f"\n[{metrica.upper()} + {wav.upper()}] Evaluando...")
            print(f"  -> Instancias balanceadas: {len(y)} | Usuarios: {sum(y==0)} | Servidores: {sum(y==1)}")
            
            datos_roc_config = {}
            
            for nombre_modelo, modelo in modelos.items():
                skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
                
                f1_scores, acc_scores = [], []
                f1_c0, f1_c1 = [], []
                y_true_oof, y_prob_oof = [], []
                cm_acumulada = np.zeros((2, 2), dtype=int)
                
                for train_idx, test_idx in skf.split(X, y):
                    X_train, X_test = X[train_idx], X[test_idx]
                    y_train, y_test = y[train_idx], y[test_idx]
                    
                    scaler = StandardScaler()
                    X_train_sc = scaler.fit_transform(X_train)
                    X_test_sc = scaler.transform(X_test)
                    
                    import sklearn
                    modelo_clon = sklearn.base.clone(modelo)
                    modelo_clon.fit(X_train_sc, y_train)
                    y_pred = modelo_clon.predict(X_test_sc)
                    
                    y_prob = modelo_clon.predict_proba(X_test_sc)[:, 1]
                    y_true_oof.extend(y_test)
                    y_prob_oof.extend(y_prob)
                    
                    acc_scores.append(accuracy_score(y_test, y_pred))
                    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1], zero_division=0)
                    f1_c0.append(f[0]); f1_c1.append(f[1])
                    f1_scores.append(np.mean([f[0], f[1]]))
                    
                    cm_acumulada += confusion_matrix(y_test, y_pred, labels=[0, 1])

                f1_media = np.mean(f1_scores)
                ci = st.t.interval(0.95, df=len(f1_scores)-1, loc=f1_media, scale=st.sem(f1_scores))
                margen_error = ci[1] - f1_media if not np.isnan(ci[1]) else 0

                resultados_globales.append({
                    "Configuracion": f"{metrica.upper()} + {wav.upper()}",
                    "Modelo": nombre_modelo.split('_', 1)[1],
                    "Accuracy": f"{np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}",
                    "F1_Global (95% CI)": f"{f1_media:.4f} ± {margen_error:.4f}",
                    "F1_Usr": f"{np.mean(f1_c0):.4f} ± {np.std(f1_c0):.4f}",
                    "F1_Srv": f"{np.mean(f1_c1):.4f} ± {np.std(f1_c1):.4f}"
                })
                
                fpr, tpr, _ = roc_curve(y_true_oof, y_prob_oof)
                roc_auc = auc(fpr, tpr)
                datos_roc_config[nombre_modelo.split('_', 1)[1]] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}
                
                # Gráficas específicas de Random Forest
                if nombre_modelo == '3_RF_Propuesto':
                    plt.figure(figsize=(6, 5))
                    sns.heatmap(cm_acumulada, annot=True, fmt='d', cmap='Greens', cbar=False,
                                xticklabels=['Usuar (0)', 'Serv (1)'], yticklabels=['Usuar (0)', 'Serv (1)'])
                    plt.title(f"Matriz de Confusión Global (10 Folds)\nRandom Forest: {metrica.upper()} + {wav.upper()}")
                    plt.xlabel("Predicción")
                    plt.ylabel("Realidad")
                    plt.tight_layout()
                    plt.savefig(os.path.join(OUTPUT_DIR, f"cm_10fold_RF_{metrica}_{wav}.pdf"), format='pdf', bbox_inches='tight')
                    plt.close()

                    # Extracción e Importancia de Gini
                    modelo_final = sklearn.base.clone(modelo)
                    X_sc_total = StandardScaler().fit_transform(X)
                    modelo_final.fit(X_sc_total, y)
                    importancias = modelo_final.feature_importances_
                    
                    df_imp = pd.DataFrame({'Caracteristica': nombres_features, 'Importancia_Gini': importancias})
                    df_imp = df_imp.sort_values(by='Importancia_Gini', ascending=False).head(15)
                    
                    plt.figure(figsize=(10, 6))
                    sns.barplot(x='Importancia_Gini', y='Caracteristica', data=df_imp, palette='YlGnBu_r')
                    plt.title(f"Top 15 Niveles Wavelet más discriminantes (Impureza Gini)\n{metrica.upper()} + {wav.upper()}")
                    plt.xlabel("Importancia Media (Disminución de impureza Gini)")
                    plt.ylabel("Nivel / Escala Wavelet")
                    plt.grid(axis='x', linestyle='--', alpha=0.7)
                    plt.tight_layout()
                    plt.savefig(os.path.join(OUTPUT_DIR, f"gini_importance_{metrica}_{wav}.pdf"), format='pdf', bbox_inches='tight')
                    plt.close()

            plt.figure(figsize=(8, 6))
            colores_roc = {'Baseline_Aleatorio': 'gray', 'Baseline_Lineal': 'darkorange', 'RF_Propuesto': 'forestgreen'}
            for mod_nombre, datos in datos_roc_config.items():
                plt.plot(datos['fpr'], datos['tpr'], lw=2, color=colores_roc.get(mod_nombre, 'blue'),
                         label=f"{mod_nombre} (AUC = {datos['auc']:.4f})")
            
            plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
            plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
            plt.xlabel('Tasa de Falsos Positivos (FPR)')
            plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
            plt.title(f'Comparativa Curva ROC: {metrica.upper()} + {wav.upper()}')
            plt.legend(loc="lower right")
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, f"roc_curve_rf_{metrica}_{wav}.pdf"), format='pdf', bbox_inches='tight')
            plt.close()

    # ==========================================
    # 4. EXPORTAR RESULTADOS
    # ==========================================
    df_resultados = pd.DataFrame(resultados_globales)
    print("\n" + "="*120)
    print("REPORTE ESTADÍSTICO FINAL - RANDOM FOREST")
    print("="*120)
    print(df_resultados.to_string(index=False))
    df_resultados.to_csv(os.path.join(OUTPUT_DIR, "reporte_mensual_rf.csv"), index=False)
    print(f"\n[INFO] Gráficas (incluyendo Importancia Gini) guardadas en: {OUTPUT_DIR}")

if __name__ == "__main__":
    ejecutar_benchmark_rf()

## Usuarios vs Servidores - Ventanas de 1 hora (Random Forest)

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import scipy.stats as st
import time
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================

# [CONFIGURACIÓN DE USUARIO]: Define las rutas de entrada y salida
BASE_DIR = " "
INPUT_DIR = os.path.join(BASE_DIR, "caracteristicas_ventanas")
OUTPUT_DIR = os.path.join(BASE_DIR, "resultados_ventanas_1hora_rf")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# [CONFIGURACIÓN DE USUARIO]: Define explícitamente las etiquetas que corresponden a los servidores.
# El resto se clasificarán automáticamente como usuarios.
ETIQUETAS_SERVIDORES = [
    'etiqueta_servidores_1', 
    'etiqueta_servidores_2'
]

NUMERO_DE_ARBOLES = 100
N_SPLITS = 10  # 10-Fold para GroupKFold

# ==========================================
# 2. FUNCIÓN DE CARGA Y BALANCEO
# ==========================================
def cargar_dataset_ventanas(metrica, wavelet):
    archivo = f"dataset_ventanas_{metrica}_{wavelet}.csv"
    ruta = os.path.join(INPUT_DIR, archivo)
    
    if not os.path.exists(ruta):
        return None, None, None, None
        
    print(f"  -> Leyendo CSV ({archivo})...")
    df = pd.read_csv(ruta)
    
    # Crear target binario
    df['target'] = df['etiqueta'].apply(lambda x: 1 if x in ETIQUETAS_SERVIDORES else 0)
    
    # Balanceo de clases (Undersampling de la mayoritaria)
    df_usuarios = df[df['target'] == 0]
    df_servidores = df[df['target'] == 1]
    
    n_min = min(len(df_usuarios), len(df_servidores))
    
    if n_min > 0:
        df_usuarios_bal = resample(df_usuarios, replace=False, n_samples=n_min, random_state=42)
        df_servidores_bal = resample(df_servidores, replace=False, n_samples=n_min, random_state=42)
        df = pd.concat([df_usuarios_bal, df_servidores_bal], ignore_index=True)
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    else:
        print("  [ERROR] Una de las clases no tiene datos suficientes para el balanceo.")
        return None, None, None, None
    
    features = [c for c in df.columns if c.startswith('nivel_') or c.startswith('escala_')]
    try:
        features.sort(key=lambda x: int(x.split('_')[1]))
    except:
        features.sort()
    
    X = df[features].values
    y = df['target'].values
    grupos_ip = df['ip'].values  # Variable crítica para agrupar en el GroupKFold
    
    return X, y, grupos_ip, features

# ==========================================
# 3. EJECUCIÓN DEL BENCHMARK ESTADÍSTICO
# ==========================================
def entrenar_ventanas_rf():
    print("=== INICIANDO BENCHMARK VENTANAS 1H (GROUP 10-FOLD + RANDOM FOREST) ===")
    
    metricas = ["energia", "entropia"]
    wavelets = ["haar", "db2", "mexicanhat"] 
    
    modelos = {
        '1_Baseline_Aleatorio': DummyClassifier(strategy='stratified', random_state=42),
        '2_Baseline_Lineal': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
        '3_RF_Propuesto': RandomForestClassifier(n_estimators=NUMERO_DE_ARBOLES, class_weight='balanced', random_state=42, n_jobs=-1)
    }

    resultados_globales = []

    for metrica in metricas:
        for wav in wavelets:
            X, y, grupos_ip, features = cargar_dataset_ventanas(metrica, wav)
            if X is None: continue
                
            print(f"\n[{metrica.upper()} + {wav.upper()}] Evaluando...")
            print(f"  -> Instancias balanceadas: {len(y)} | Usuarios: {sum(y==0)} | Servidores: {sum(y==1)}")
            
            datos_roc_config = {}
            importancias_acumuladas = np.zeros(len(features))
            
            for nombre_modelo, modelo in modelos.items():
                t0 = time.time()
                gkf = GroupKFold(n_splits=N_SPLITS)
                
                f1_scores, acc_scores = [], []
                f1_c0, f1_c1 = [], []
                y_true_oof, y_prob_oof = [], []
                cm_acumulada = np.zeros((2, 2), dtype=int)
                
                # Bucle de GroupKFold (asegura que las IPs enteras caigan en train o test)
                for train_idx, test_idx in gkf.split(X, y, groups=grupos_ip):
                    X_tr, y_tr = X[train_idx], y[train_idx]
                    X_te, y_te = X[test_idx], y[test_idx]
                    
                    scaler = StandardScaler()
                    X_tr_sc = scaler.fit_transform(X_tr)
                    X_te_sc = scaler.transform(X_te)
                    
                    import sklearn
                    modelo_clon = sklearn.base.clone(modelo)
                    modelo_clon.fit(X_tr_sc, y_tr)
                    y_pred = modelo_clon.predict(X_te_sc)
                    
                    if nombre_modelo == '3_RF_Propuesto':
                        importancias_acumuladas += modelo_clon.feature_importances_ / N_SPLITS
                    
                    y_prob = modelo_clon.predict_proba(X_te_sc)[:, 1]
                    y_true_oof.extend(y_te)
                    y_prob_oof.extend(y_prob)
                    
                    acc_scores.append(accuracy_score(y_te, y_pred))
                    p, r, f, _ = precision_recall_fscore_support(y_te, y_pred, labels=[0, 1], zero_division=0)
                    
                    f1_c0.append(f[0]); f1_c1.append(f[1])
                    f1_scores.append(np.mean([f[0], f[1]]))
                    cm_acumulada += confusion_matrix(y_te, y_pred, labels=[0, 1])

                f1_media = np.mean(f1_scores)
                ci = st.t.interval(0.95, df=len(f1_scores)-1, loc=f1_media, scale=st.sem(f1_scores))
                margen_error = ci[1] - f1_media if not np.isnan(ci[1]) else 0

                resultados_globales.append({
                    "Configuracion": f"{metrica.upper()} + {wav.upper()}",
                    "Modelo": nombre_modelo.split('_', 1)[1],
                    "Accuracy": f"{np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}",
                    "F1_Global (95% CI)": f"{f1_media:.4f} ± {margen_error:.4f}",
                    "F1_Usr": f"{np.mean(f1_c0):.4f} ± {np.std(f1_c0):.4f}",
                    "F1_Srv": f"{np.mean(f1_c1):.4f} ± {np.std(f1_c1):.4f}"
                })
                
                fpr, tpr, _ = roc_curve(y_true_oof, y_prob_oof)
                roc_auc = auc(fpr, tpr)
                datos_roc_config[nombre_modelo.split('_', 1)[1]] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}
                
                if nombre_modelo == '3_RF_Propuesto':
                    plt.figure(figsize=(6, 5))
                    sns.heatmap(cm_acumulada, annot=True, fmt='d', cmap='YlOrBr', cbar=False,
                                xticklabels=['Usuar (0)', 'Serv (1)'], yticklabels=['Usuar (0)', 'Serv (1)'])
                    plt.title(f"Matriz de Confusión Ventanas 1H (Group 10-Fold)\nRF: {metrica.upper()} + {wav.upper()}")
                    plt.xlabel("Predicción")
                    plt.ylabel("Realidad")
                    plt.tight_layout()
                    plt.savefig(os.path.join(OUTPUT_DIR, f"cm_ventanas_RF_{metrica}_{wav}.pdf"), format='pdf', bbox_inches='tight')
                    plt.close()
                    
                    indices = np.argsort(importancias_acumuladas)[::-1]
                    plt.figure(figsize=(10, 6))
                    sns.barplot(x=importancias_acumuladas[indices], y=np.array(features)[indices], palette="viridis")
                    plt.title(f"Top Características Diferenciadoras (Ventanas 1h)\nRF: {metrica.upper()} + {wav.upper()}")
                    plt.xlabel("Importancia Media (Reducción de Impureza de Gini)")
                    plt.ylabel("Niveles Wavelet")
                    plt.tight_layout()
                    plt.savefig(os.path.join(OUTPUT_DIR, f"feature_importance_RF_{metrica}_{wav}.pdf"), format='pdf', bbox_inches='tight')
                    plt.close()
                
                print(f"    - {nombre_modelo.split('_', 1)[1]} procesado en {round(time.time()-t0, 2)}s")

            plt.figure(figsize=(8, 6))
            colores_roc = {'Baseline_Aleatorio': 'gray', 'Baseline_Lineal': 'darkorange', 'RF_Propuesto': 'forestgreen'}
            for mod_nombre, datos in datos_roc_config.items():
                plt.plot(datos['fpr'], datos['tpr'], lw=2, color=colores_roc.get(mod_nombre, 'blue'),
                         label=f"{mod_nombre} (AUC = {datos['auc']:.4f})")
            
            plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
            plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
            plt.xlabel('Tasa de Falsos Positivos (FPR)')
            plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
            plt.title(f'Comparativa Curva ROC Ventanas 1H: {metrica.upper()} + {wav.upper()}')
            plt.legend(loc="lower right")
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, f"roc_curve_ventanas_{metrica}_{wav}.pdf"), format='pdf', bbox_inches='tight')
            plt.close()

    # ==========================================
    # 4. EXPORTAR RESULTADOS 
    # ==========================================
    df_resultados = pd.DataFrame(resultados_globales)
    print("\n" + "="*120)
    print("REPORTE ESTADÍSTICO FINAL VENTANAS 1H (GROUP 10-FOLD)")
    print("="*120)
    print(df_resultados.to_string(index=False))
    
    df_resultados.to_csv(os.path.join(OUTPUT_DIR, "reporte_ventanas_rf.csv"), index=False)
    print(f"\n[INFO] Gráficas y tablas finales guardadas en: {OUTPUT_DIR}")

if __name__ == "__main__":
    entrenar_ventanas_rf()